# Fairness Audit Demo - FairML Consulting
This notebook demonstrates the capabilities of the **Measurement Module** for auditing bias in loan approval datasets.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analyzer import FairnessAnalyzer, FairnessResult

# Setting plot style for better visualizations later
plt.style.use('ggplot')
%matplotlib inline

# Load the dataset
file_path = 'Loan Dataset.csv'
df_raw = pd.read_csv(file_path)

# Clean column names just in case there are leading/trailing spaces
df_raw.columns = df_raw.columns.str.strip()

# Display the first few rows and column info to verify the schema
print(f"Dataset shape: {df_raw.shape}")
display(df_raw.head())
print(df_raw.info())

print("Setup complete. FairnessAnalyzer imported.")

ImportError: cannot import name 'FairnessResult' from 'analyzer' (c:\Users\gloccioni\AI-Ethics-Project\analyzer.py)

In [21]:
# Demonstrate Binning for Age
age_bins = [0, 25, 35, 45, 55, 65, np.inf]
age_labels = ['0-25', '26-35', '36-45', '46-55', '56-65', '65+']

analyzer.bin_column('Age', bins=age_bins, labels=age_labels, new_column_name='Age_Group')

# Show distribution
analyzer.get_group_stats('Loan_Approval_Status', 'Age_Group')

Column 'Age' binned into 'Age_Group' with labels: ['0-25', '26-35', '36-45', '46-55', '56-65', '65+']


c:\Users\gloccioni\AI-Ethics-Project\analyzer.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = self.df.groupby(sensitive_col).size()


Age_Group
0-25      3889
26-35    17448
36-45    17835
46-55     8716
56-65     2966
65+       1146
dtype: int64

In [17]:
# Configuration using your specific target and positive outcome
target_column = 'Loan_Approval_Status'
positive_outcome = 1 
sensitive_feature = 'Age_Group'

# 1. Calculate Selection Rates
selection_rates = analyzer.calculate_selection_rate(target_column, sensitive_feature, positive_outcome)

print("--- Selection Rates (Approval Rates) per Age Group ---")
print(selection_rates)

# 2. Calculate Demographic Parity Ratio
parity_ratio = analyzer.calculate_demographic_parity(selection_rates)

print("-" * 30)
print(f"Demographic Parity Ratio: {parity_ratio:.4f}")
print("-" * 30)

# 3. Final Evaluation
if parity_ratio < 0.8:
    print("STATUS: Bias detected (Below 0.8 threshold).")
else:
    print("STATUS: No significant bias detected (Above 0.8 threshold).")

--- Selection Rates (Approval Rates) per Age Group ---
Age_Group
0-25     0.419388
26-35    0.730399
36-45    0.731259
46-55    0.609224
56-65    0.160148
65+      0.143106
Name: Loan_Approval_Status, dtype: float64
------------------------------
Demographic Parity Ratio: 0.1957
------------------------------
STATUS: Bias detected (Below 0.8 threshold).


c:\Users\gloccioni\AI-Ethics-Project\analyzer.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  selection_rates = self.df.groupby(sensitive_col)[target_col].apply(


### Ethical Assessment
The fairness audit reveals a **significant Disparate Impact** against younger and older age groups:
* **Primary Bias:** Applicants aged **56-65** and **65+** show approval rates below 16%, compared to the 73% of the **26-45** group.
* **Metric:** The Demographic Parity Ratio of **0.1957** is well below the **0.8** industry standard, indicating that the loan approval process is not fair regarding the applicant's age.
* **Conclusion:** The model requires mitigation to reduce the bias against elderly applicants.